# LIMPIEZA DE DATA PARA TRUSTED

## Inicialización de Spark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_timestamp

spark = SparkSession.builder.appName("trusted_dataset_cleaning").getOrCreate()

## Ruta de entrada (CSV crudo en raw) y Leer archivo CSV desde raw

In [5]:
path_raw = "gs://retail-transactions-final/raw/Retail_Transactions_Dataset.csv"

df_raw = spark.read.option("header", True).csv(path_raw)

## Ver esquema inicial

In [6]:
df_raw.printSchema()
df_raw.show(5)

root
 |-- Transaction_ID: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: string (nullable = true)
 |-- Total_Cost: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: string (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)

+--------------+-------------------+-----------------+--------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------+
|Transaction_ID|               Date|    Customer_Name|             Product|Total_Items|Total_Cost|Payment_Method|         City|      Store_Type|Discount_Applied|Customer_Category|Season|           Promotion|
+--------------+-----------

## Limpieza

In [7]:
# 1. Convertir columnas numéricas
df_cleaned = df_raw \
    .withColumn("Total_Items", col("Total_Items").cast("int")) \
    .withColumn("Total_Cost", col("Total_Cost").cast("double")) \
    .withColumn("Transaction_ID", col("Transaction_ID").cast("long"))

# 2. Convertir Date a timestamp
df_cleaned = df_cleaned.withColumn("Date", to_timestamp(col("Date"), "yyyy-MM-dd HH:mm:ss"))

# 3. Reemplazar nulos en 'Promotion' con 'None'
df_cleaned = df_cleaned.withColumn("Promotion", when(col("Promotion").isNull(), "None").otherwise(col("Promotion")))

# 4. Opcional: validar si 'Product' parece lista (lo dejamos como string por ahora)
df_cleaned.select("Product").show(5, truncate=False)

+-------------------------------------------------------+
|Product                                                |
+-------------------------------------------------------+
|['Ketchup', 'Shaving Cream', 'Light Bulbs']            |
|['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'Potatoes']|
|['Spinach']                                            |
|['Tissues', 'Mustard']                                 |
|['Dish Soap']                                          |
+-------------------------------------------------------+
only showing top 5 rows



## Verificamos el esquema final

In [8]:
df_cleaned.printSchema()

root
 |-- Transaction_ID: long (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: string (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)



## Guardar en formato Parquet en la capa trusted

In [9]:
path_trusted = "gs://retail-transactions-final/trusted/retail_trusted.parquet"
df_cleaned.write.mode("overwrite").parquet(path_trusted)
print("✅ Dataset limpio guardado en trusted como Parquet.")

✅ Dataset limpio guardado en trusted como Parquet.
